# Notebook 04: ArcGIS Web Maps and Dashboard

**Publishing the Lake Mead near-shore clustering results to ArcGIS Online**

In this notebook, I took part of the clustering results from Notebook 03 and published them as an interactive comparison on ArcGIS Online. I focused on the near-shore subset (214 fires within 5 km of the lake), since that's where the two methods differ most: obstacle-aware $k$-means with optimized $\beta$ improves arc-length span by about 19% over standard $k$-means, and with fewer fires the individual cluster shifts are easy to see.

I pulled in four files saved from Notebook 03:

- `mead_fires_clustered.csv`: all 844 wildfires with cluster labels from each method
- `mead_cluster_summary.csv`: per-cluster stats for the near-shore subset
- `mead_global_metrics.csv`: the headline near-shore metrics
- `mead_boundary.csv`: the cleaned Lake Mead shoreline polygon

From here I prepped the near-shore data, connected to my ArcGIS Online account, and published four hosted items: the fires feature layer, two supporting tables, and the boundary polygon. Those became the data sources for two web maps and a comparison dashboard, which I built in the ArcGIS Online interface. Links are provided at the end of the notebook.

In [ ]:
# Standard packages
import getpass
from pathlib import Path

import numpy as np
import pandas as pd
import shutil
import geopandas as gpd
from shapely.geometry import Polygon

# ArcGIS Python API
from arcgis.gis import GIS

## 2. Loading Results from Notebook 03

I started by loading the files saved in Notebook 03. The fires CSV holds all 844 fires (basin-wide) with their cluster labels, so I filtered it down to the near-shore subset before publishing. The cluster summary and global metrics were already saved with near-shore values in Notebook 03.

In [ ]:
processed_dir = Path('../data/processed')

fires = pd.read_csv(processed_dir / 'mead_fires_clustered.csv')
cluster_summary = pd.read_csv(processed_dir / 'mead_cluster_summary.csv')
global_metrics = pd.read_csv(processed_dir / 'mead_global_metrics.csv')

print(f'Loaded {len(fires)} fires (full basin)')

## 3. Preparing Data for ArcGIS

The upload only needs the near-shore fires (within 5 km of the shore) with their near-shore cluster labels, so before publishing I:

- filtered to the near-shore subset using `dist_to_lake_km`
- pulled `cluster_near_std` and `cluster_near_opt` into cleaner `Cluster_Std` and `Cluster_OA` columns
- added readable cause and fire-size labels
- cast the year to a string so ArcGIS wouldn't render it as "2,003"

In [ ]:
# Filtering to near-shore fires
threshold_km = 5
near_mask = fires['dist_to_lake_km'] < threshold_km
fires_pub = fires[near_mask].copy().reset_index(drop=True)

print(f'Near-shore subset: {len(fires_pub)} fires within {threshold_km} km of Lake Mead')

# 1-indexed cluster labels (the dashboard displays Cluster 1, 2, 3, 4)
fires_pub['Cluster_Std'] = (fires_pub['cluster_near_std'] + 1).astype(int)
fires_pub['Cluster_OA'] = (fires_pub['cluster_near_opt'] + 1).astype(int)

# Adding human-caused binary
fires_pub['Is_Human_Caused'] = fires_pub['cause_binary'].astype(int)

# Cause labels
fires_pub['Cause'] = fires_pub['cause_binary'].map({0: 'Natural', 1: 'Human-Caused'})
fires_pub['Specific_Cause'] = fires_pub['NWCG_GENERAL_CAUSE'].replace('Natural', 'Lightning')

# Rounding fire size for popup readability
fires_pub['Fire_Size_Acres'] = fires_pub['FIRE_SIZE'].round(2)

# Casting year to string so ArcGIS doesn't render it as "2,003"
fires_pub['Fire_Year'] = fires_pub['FIRE_YEAR'].astype(int).astype(str)

# Distance to lake 
fires_pub['Dist_To_Lake_Km'] = fires_pub['dist_to_lake_km'].round(2)

# Columns to publish
publish_cols = [
    'LATITUDE', 'LONGITUDE',
    'Cluster_Std',
    'Cluster_OA',
    'Cause', 'Specific_Cause',
    'Is_Human_Caused', 
    'Fire_Size_Acres', 'Fire_Year',
    'Dist_To_Lake_Km',
]
fires_pub = fires_pub[publish_cols].copy()

# Saving the publishing-ready CSV
publish_path = processed_dir / 'mead_fires_nearshore_webmap.csv'
fires_pub.to_csv(publish_path, index=False)

print(f'\nSaved {len(fires_pub)} near-shore fires to {publish_path.name}')

## 4. Connecting to ArcGIS Online

I connected to my ArcGIS Online organization through the ArcGIS Python API, authenticating with a password prompt.

In [ ]:
gis = GIS(
    url='https://michele-75.maps.arcgis.com',
    username='Michele-75',
    password=getpass.getpass('Enter your ArcGIS Online password: '),
)

print(f'Connected as: {gis.users.me.username}')
print(f'Organization: {gis.properties.name}')

## 5. Publishing the Hosted Items

I published four hosted items to ArcGIS Online:

1. **Near-shore fires feature layer** -- 214 points, one per fire, with cluster columns for both methods.
2. **Cluster summary table** -- 8 rows (4 per method) with per-cluster stats for the dashboard side panels.
3. **Global metrics table** -- 1 row with the headline numbers, including the 19% span improvement.
4. **Lake Mead boundary feature layer** -- the simplified shoreline polygon, hosted so it can sit under both web maps as a reference.

To avoid duplicates on a re-run, the cell below skips items that already exist by default; setting `delete_existing` to `True` deletes and recreates them cleanly.

In [ ]:
delete_existing = False

# Titles for items
item_titles = [
    'Lake Mead Near-Shore Wildfires - Obstacle-Aware vs Standard k-Means',
    'Lake Mead Near-Shore Wildfires - Cluster Summary',
    'Lake Mead Near-Shore Wildfires - Global Metrics',
    'Lake Mead Boundary',
]

if delete_existing:
    me = gis.users.me.username
    for title in item_titles:
        existing = gis.content.search(
            query=f'title:"{title}" owner:{me}',
            max_items=20,
        )
        existing = [it for it in existing if it.title == title]
        for it in existing:
            print(f'Deleting old item: {it.title} ({it.type}, ID {it.id})')
            it.delete()
    print('Cleanup complete.\n')
else:
    print('Skipping cleanup. New items will be created alongside any existing duplicates.\n')

### 5.1 Publishing the Fires Feature Layer

In [ ]:
# Item properties for the CSV (the resulting feature layer inherits these)
fire_item_properties = {
    'title': 'Lake Mead Near-Shore Wildfires - Obstacle-Aware vs Standard k-Means',
    'description': (
        'Wildfire occurrence data (1992-2020) within 5 km of Lake Mead\'s shoreline, '
        'clustered two ways for direct comparison: '
        '<b>standard k-Means</b> (geographic coordinates only) and '
        '<b>obstacle-aware k-Means</b> (geographic coordinates plus arc-length parameter '
        '<i>s</i> along the Lake Mead shoreline). Each fire has two cluster labels, '
        'one per method, so a dashboard can compare the two clusterings side by side.<br><br>'
        '<b>Source:</b> FPA FOD 6th Edition (USFS Research Data Archive).<br>'
        '<b>Method:</b> Obstacle-aware k-Means with optimized weight β selected '
        'from the J / span objective surface within the near-shore subset. See project '
        'repository for full methodology.'
    ),
    'tags': 'wildfires, clustering, Lake Mead, near-shore, obstacle-aware, k-means, portfolio',
    'type': 'CSV',
}

print('Uploading near-shore fire data to ArcGIS Online...')
fire_csv_item = gis.content.add(fire_item_properties, data=str(publish_path))
print(f'CSV uploaded: {fire_csv_item.title} (ID: {fire_csv_item.id})')

print('\nPublishing as hosted feature layer...')
fire_layer_item = fire_csv_item.publish()
print(f'Published: {fire_layer_item.title}')
print(f'Item URL: {fire_layer_item.homepage}')

### 5.2 Publishing the Supporting Tables

In [ ]:
# Cluster summary table
summary_item_props = {
    'title': 'Lake Mead Near-Shore Wildfires - Cluster Summary',
    'description': (
        'Per-cluster summary statistics for the near-shore Lake Mead clustering, with '
        '4 rows for standard k-Means (method = "std") and 4 rows for obstacle-aware '
        'optimized (method = "oa_opt"). Includes n_fires, mean and median fire size, '
        'percent human-caused, and per-cluster arc-length span. Used to populate the '
        'dashboard side panels.'
    ),
    'tags': 'wildfires, clustering, Lake Mead, near-shore, summary, dashboard',
    'type': 'CSV',
}
summary_csv_path = processed_dir / 'mead_cluster_summary.csv'
summary_csv_item = gis.content.add(summary_item_props, data=str(summary_csv_path))
summary_table_item = summary_csv_item.publish()
print(f'Cluster summary published: {summary_table_item.title}')
print(f'  {summary_table_item.homepage}')

# Global metrics table
metrics_item_props = {
    'title': 'Lake Mead Near-Shore Wildfires - Global Metrics',
    'description': (
        'Headline metrics for the near-shore Lake Mead clustering. Single row with '
        'total fire count, year range, optimal β, mean arc-length span for both '
        'methods, and the percent span improvement. Used to populate the dashboard '
        'headline indicators.'
    ),
    'tags': 'wildfires, clustering, Lake Mead, near-shore, metrics, dashboard',
    'type': 'CSV',
}
metrics_csv_path = processed_dir / 'mead_global_metrics.csv'
metrics_csv_item = gis.content.add(metrics_item_props, data=str(metrics_csv_path))
metrics_table_item = metrics_csv_item.publish()
print(f'Global metrics published: {metrics_table_item.title}')
print(f'  {metrics_table_item.homepage}')

### 5.3 Publishing the Lake Mead Boundary Polygon

I published the shoreline polygon as its own hosted layer so I could easily add it to the webmaps. To get it into AGOL cleanly, I converted the boundary CSV into a zipped shapefile (reliable format for polygon imports), uploaded it, and published it as a feature layer.

In [ ]:
# Building a Shapely polygon from the saved boundary CSV
boundary_df = pd.read_csv(Path('../data/boundaries/mead_boundary.csv'))
coords = list(zip(boundary_df['longitude'], boundary_df['latitude']))
polygon = Polygon(coords)

gdf = gpd.GeoDataFrame(
    {'name': ['Lake Mead']},
    geometry=[polygon],
    crs='EPSG:4326'   # WGS 84
)

# Saving to a shapefile 
shp_dir = Path('../data/boundaries/mead_boundary_shp')
shp_dir.mkdir(exist_ok=True)
gdf.to_file(shp_dir / 'mead_boundary.shp')

# Zip the shapefile directory for AGOL upload
zip_path = Path('../data/boundaries/mead_boundary')
shutil.make_archive(str(zip_path), 'zip', shp_dir)
zip_file = Path(f'{zip_path}.zip')
print(f'Saved zipped shapefile to {zip_file}')

# Uploading as an AGOL item and publish as a hosted feature layer
boundary_item_props = {
    'title': 'Lake Mead Boundary',
    'description': (
        'Simplified shoreline polygon for Lake Mead, derived from the USGS National '
        'Hydrography Dataset (Waterbody layer 12). Built by unioning all NHD features '
        'above 1 sq km named "Lake Mead" to capture the main body plus the eastern '
        'arm, then simplifying with Douglas-Peucker (tolerance 0.005 degrees). Used '
        'as the boundary for obstacle-aware k-Means clustering and as a visual '
        'reference layer on the comparison dashboard.'
    ),
    'tags': 'Lake Mead, boundary, NHD, shoreline, portfolio',
    'type': 'Shapefile',
}

print('\nUploading boundary shapefile to ArcGIS Online...')
boundary_zip_item = gis.content.add(boundary_item_props, data=str(zip_file))
print(f'Shapefile uploaded: {boundary_zip_item.title} (ID: {boundary_zip_item.id})')

print('\nPublishing as hosted feature layer...')
boundary_layer_item = boundary_zip_item.publish()
print(f'Published: {boundary_layer_item.title}')
print(f'Item URL: {boundary_layer_item.homepage}')

## 6. The Web Maps and Dashboard

Using the four hosted items published above as data sources, I built two web maps and a comparison dashboard in the ArcGIS Online interface.

The two web maps use the same near-shore fires layer, each styled by a different cluster column over the Lake Mead boundary polygon: one colors fires by their standard $k$-means cluster, the other by their obstacle-aware cluster. The dashboard places the two maps side by side with a shared header, the 19% span-improvement headline, and per-cluster panels (fire counts, mean fire size, cause breakdown, and arc-length span), so the two clusterings can be compared directly.

- **Comparison dashboard**: [https://michele-75.maps.arcgis.com/home/item.html?id=b0a0e3a1258440ba86c61569bfea2185]
- **Standard k-means web map**: [https://michele-75.maps.arcgis.com/home/item.html?id=e68e97a9f81b446abaf1b5433306f4da]
- **Obstacle-aware web map**: [https://michele-75.maps.arcgis.com/home/item.html?id=2351b66ac503433ebdc30e1e8b4154f1]